In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ppi_py import ppi_logistic_ci
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("data/webgpt_sonnet_5k.csv")
mask = df["yhat"].notna()
df = df[mask].reset_index(drop=True)

# Covariate: difference in response length (len(answer_1) - len(answer_0))
len_0 = df["answer_0"].fillna("").str.len().values.astype(float)
len_1 = df["answer_1"].fillna("").str.len().values.astype(float)
length_diff = len_1 - len_0

# Standardize for numerical stability
length_diff_mean = length_diff.mean()
length_diff_std = length_diff.std()
X_raw = (length_diff - length_diff_mean) / length_diff_std

# Design matrix: intercept + standardized length difference
X = np.column_stack([np.ones(len(df)), X_raw])

# Binarize: Y=1 if voter preferred answer B (vote > 0.5)
Y = (df["vote"].values > 0.5).astype(int)
Yhat = df["yhat"].values  # predicted probabilities

# Permute data so the fixed-split table is not biased by row order
rng0 = np.random.default_rng(42)
perm = rng0.permutation(len(Y))
X, Y, Yhat = X[perm], Y[perm], Yhat[perm]

print(f"N = {len(df)}")
print(f"Columns: intercept, standardized length_diff(B-A)")
print(f"Y is binary: unique values = {np.unique(Y)}")
df[["question", "vote", "yhat"]].head()

N = 5000
Columns: intercept, standardized length_diff(B-A)
Y is binary: unique values = [0 1]


,question,vote,yhat
0,"Voiced by Harry Shearer, what Simpsons charact...",0.00,0.200
1,Alliumphobia is the irrational fear of which p...,0.50,0.250
2,Heterophobia is the irrational fear of what,0.25,0.900
3,"What was the name of Dan Dare's co-pilot, in t...",0.50,0.675
4,"In 1965, which Christmas song became the first...",0.50,0.650


In [2]:
# Single-split demo at n=100
n = 100
alpha = 0.05
coord = 1  # the length_diff coefficient

X_l, X_u = X[:n], X[n:]
Y_l, Yh_l, Yh_u = Y[:n], Yhat[:n], Yhat[n:]

classical_ci = ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=0)
ppi_ci = ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=1)
ppi_pp_ci = ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha)

all_cis = [("Classical", classical_ci), ("PPI", ppi_ci), ("PPI++", ppi_pp_ci)]
classical_width = float(classical_ci[1][coord]) - float(classical_ci[0][coord])

print(f"Logistic regression coefficient (length_diff -> vote), n={n}")
print()
print(f"{'Method':<12} {'CI':>22}   {'Width':>8}   {'vs Classical':>12}")
print("-" * 60)
for name, ci in all_cis:
    lo, hi = float(ci[0][coord]), float(ci[1][coord])
    w = hi - lo
    reduction = (1 - w / classical_width) * 100
    ci_str = f"[{lo:.4f}, {hi:.4f}]"
    red_str = "---" if name == "Classical" else f"{reduction:+.1f}%"
    print(f"{name:<12} {ci_str:>22}   {w:>8.4f}   {red_str:>12}")

Logistic regression coefficient (length_diff -> vote), n=100

Method                           CI      Width   vs Classical
------------------------------------------------------------
Classical          [0.0435, 0.8657]     0.8222            ---
PPI                [0.0583, 1.0222]     0.9639         -17.2%
PPI++              [0.0728, 0.9028]     0.8299          -0.9%


In [3]:
# Width and coverage as a function of n
alpha = 0.05
n_trials = 200
ns = np.arange(50, 520, 20)
coord = 1  # length_diff coefficient

# "True" coefficient from full-data logistic regression
lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=10000, tol=1e-15, fit_intercept=False)
lr.fit(X, Y)
beta_true = lr.coef_.squeeze()[coord]
print(f"Full-data logistic regression coefficient for length_diff: {beta_true:.4f}")

widths = {m: np.zeros(len(ns)) for m in ["Classical", "PPI", "PPI++"]}
covers = {m: np.zeros(len(ns)) for m in ["Classical", "PPI", "PPI++"]}

rng = np.random.default_rng(42)

for j, n in enumerate(ns):
    w_acc = {m: 0.0 for m in widths}
    c_acc = {m: 0 for m in widths}
    n_valid = 0
    for t in range(n_trials):
        idx = rng.permutation(len(Y))
        X_l, X_u = X[idx[:n]], X[idx[n:]]
        Y_l = Y[idx[:n]]
        Yh_l, Yh_u = Yhat[idx[:n]], Yhat[idx[n:]]

        # Skip degenerate samples (e.g. all-0 or all-1)
        if len(np.unique(Y_l)) < 2:
            continue

        try:
            cis = [
                ("Classical", ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=0)),
                ("PPI", ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=1)),
                ("PPI++", ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha)),
            ]
        except np.linalg.LinAlgError:
            continue

        n_valid += 1
        for name, ci in cis:
            lo, hi = float(ci[0][coord]), float(ci[1][coord])
            w_acc[name] += hi - lo
            c_acc[name] += int(lo <= beta_true <= hi)

    for m in widths:
        widths[m][j] = w_acc[m] / n_valid if n_valid > 0 else np.nan
        covers[m][j] = c_acc[m] / n_valid if n_valid > 0 else np.nan
    if (j + 1) % 5 == 0:
        print(f"  n={n} done ({n_valid}/{n_trials} valid trials)")

Full-data logistic regression coefficient for length_diff: 0.3656
  n=130 done (200/200 valid trials)
  n=230 done (200/200 valid trials)
  n=330 done (200/200 valid trials)
  n=430 done (200/200 valid trials)


In [4]:
# Table: average CI width and improvement over Classical at each n
rows = []
for j, n in enumerate(ns):
    w_cl = widths["Classical"][j]
    w_ppi = widths["PPI"][j]
    w_pp = widths["PPI++"][j]
    rows.append({
        "n": int(n),
        "Classical": f"{w_cl:.4f}",
        "PPI": f"{w_ppi:.4f}",
        "PPI++ ": f"{w_pp:.4f}",
        "PPI vs Classical": f"{(1 - w_ppi / w_cl) * 100:+.1f}%",
        "PPI++ vs Classical": f"{(1 - w_pp / w_cl) * 100:+.1f}%",
    })

table_df = pd.DataFrame(rows).set_index("n")
print(table_df.to_string())

    Classical     PPI  PPI++  PPI vs Classical PPI++ vs Classical
n                                                                
50     1.3210  1.3543  1.1779            -2.5%             +10.8%
70     1.0602  1.1208  0.9951            -5.7%              +6.1%
90     0.9363  0.9669  0.8664            -3.3%              +7.5%
110    0.8481  0.8815  0.7883            -3.9%              +7.1%
130    0.7730  0.7802  0.7139            -0.9%              +7.6%
150    0.7298  0.7642  0.6880            -4.7%              +5.7%
170    0.6829  0.6915  0.6351            -1.3%              +7.0%
190    0.6384  0.6552  0.5974            -2.6%              +6.4%
210    0.6075  0.6228  0.5677            -2.5%              +6.6%
230    0.5828  0.5922  0.5445            -1.6%              +6.6%
250    0.5552  0.5679  0.5229            -2.3%              +5.8%
270    0.5317  0.5458  0.5001            -2.7%              +5.9%
290    0.5120  0.5253  0.4823            -2.6%              +5.8%
310    0.4

In [ ]:
colors = {"Classical": "#6394EE", "PPI": "#84C87F", "PPI++": "#FF8B00"}
labels = {"Classical": "classical", "PPI": "PPI", "PPI++": "PPI++"}
method_order = ["PPI", "PPI++", "Classical"]

# Example intervals for the third panel
n_ex = 100
n_examples = 3
coord = 1
rng_ex = np.random.default_rng(7)

example_cis = {"Classical": [], "PPI": [], "PPI++": []}
for _ in range(n_examples):
    idx = rng_ex.permutation(len(Y))
    X_l, X_u = X[idx[:n_ex]], X[idx[n_ex:]]
    Y_l = Y[idx[:n_ex]]
    Yh_l, Yh_u = Yhat[idx[:n_ex]], Yhat[idx[n_ex:]]
    for name, ci in [
        ("Classical", ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=0)),
        ("PPI", ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha, lam=1)),
        ("PPI++", ppi_logistic_ci(X_l, Y_l, Yh_l, X_u, Yh_u, alpha=alpha)),
    ]:
        example_cis[name].append((float(ci[0][coord]), float(ci[1][coord])))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Coverage (left)
for m in method_order:
    axes[0].plot(ns, covers[m], label=labels[m], color=colors[m], linewidth=2)
axes[0].axhline(1 - alpha, color="gray", linestyle="dotted", linewidth=1.5)
axes[0].set_xlabel("n", fontsize=14)
axes[0].set_ylabel("")
axes[0].set_title("coverage", fontsize=16)
axes[0].set_ylim(0.55, 1.02)
axes[0].tick_params(axis='both', labelsize=13)
axes[0].yaxis.set_major_locator(plt.MaxNLocator(3))

# Width (middle)
for m in method_order:
    axes[1].plot(ns, widths[m], label=labels[m], color=colors[m], linewidth=2)
axes[1].set_xlabel("n", fontsize=14)
axes[1].set_ylabel("")
axes[1].set_title("width", fontsize=16)
axes[1].legend(fontsize=13)
axes[1].tick_params(axis='both', labelsize=13)
axes[1].yaxis.set_major_locator(plt.MaxNLocator(3))

# Example intervals (right)
y_pos = 0
yticks, ytick_labels = [], []
gap = 0.6
for m in method_order:
    for i, (lo, hi) in enumerate(example_cis[m]):
        axes[2].barh(y_pos, hi - lo, left=lo, height=0.5, color=colors[m], alpha=0.8,
                      edgecolor="white", linewidth=0.5)
        y_pos += 1
    yticks.append(y_pos - n_examples / 2)
    ytick_labels.append(labels[m])
    y_pos += gap

axes[2].axvline(beta_true, color="black", linestyle="--", linewidth=1.2, label=r"$\hat\beta_{\mathrm{full}}$")
axes[2].set_yticks(yticks)
axes[2].set_yticklabels(ytick_labels, fontsize=13)
axes[2].set_xlabel("coefficient", fontsize=14)
axes[2].set_title(f"example intervals (n={n_ex})", fontsize=16)
axes[2].tick_params(axis='x', labelsize=13)
axes[2].legend(fontsize=13)
axes[2].xaxis.set_major_locator(plt.MaxNLocator(3))

sns.despine(top=True, right=True)
plt.tight_layout()
os.makedirs('./plots/', exist_ok=True)
plt.savefig('./plots/length_vote.pdf')